# POSE GENERATION

## Renitialise Directories for Current Cycle

In [ ]:
target_name = "A71EV2AZ"  # Change this to match the target name in Fragalysis.
cycle_number = 1  # Increment this for each new design cycle for this target.

In [ ]:

# all XChem-FFF work is kept under $HOME2/XChem-FFF; each target gets its own
# subdirectory here, and each design cycle its own subdirectory within that
xchem_fff_dir = Path(environ["HOME2"]) / "XChem-FFF"
target_dir = xchem_fff_dir / target_name.lower()
cycle_dir = target_dir / f"cycle_{cycle_number:02}"
fragmenstein_dir = cycle_dir / "fragmenstein"
knitwork_dir = cycle_dir / "knitwork"
knitwork_pure_output_dir = knitwork_dir / "knitwork_pure_output"
knitwork_impure_output_dir = knitwork_dir / "knitwork_impure_output"
gnina_dir = cycle_dir / "gnina"
gnina_inputs_dir = gnina_dir / "inputs"
gnina_inputs_fragmenstein_dir =  gnina_inputs_dir / "fragmenstein"
gnina_inputs_knitwork_pure_dir =  gnina_inputs_dir / "knitwork_pure"
gnina_inputs_knitwork_impure_dir =  gnina_inputs_dir / "knitwork_impure"
gnina_outputs_dir = gnina_dir / "outputs"
gnina_outputs_fragmenstein_dir =  gnina_outputs_dir / "fragmenstein"
gnina_outputs_knitwork_pure_dir =  gnina_outputs_dir / "knitwork_pure"
gnina_outputs_knitwork_impure_dir =  gnina_outputs_dir / "knitwork_impure"
moccassin_outputs_dir = cycle_dir / "moccassin_outputs"
bulk_targets_dir = bulk_targets_dir = Path(environ["BULK"]) / "TARGETS"
bulk_target_dir = bulk_targets_dir / target_name



## Run BulkDock Placement

In [ ]:
print(f"target_name = {target_name}")
print(f"fragmenstein_dir = {fragmenstein_dir.resolve()}")
print(f"knitwork_dir = {knitwork_dir.resolve()}")

## Generate BulkDock inputs

#### __In Terminal:__

Fragmenstein scaffolds:
```bash

cd <fragmenstein_dir printed above>

python $HOME2/slurm/fragmenstein_to_bulkdock.py

cp fragmenstein_bulkdock_input.csv $BULK/INPUTS/<target_name>_fragmenstein.csv
```

Knitwork scaffolds:

File structure should be :
```
<knitwork_dir>/knitwork_pure_output/<target_name lowercase>_pure_merges.sdf
<knitwork_dir>/knitwork_impure_output/<target_name lowercase>_impure_merges.sdf
```
```bash
cd <knitwork_dir printed above>

python $HOME2/slurm/knitwork_SDF_to_bulkdock.py 

cp knitwork_pure_bulkdock_input.csv $BULK/INPUTS/<target_name>_pure_knitwork.csv
cp knitwork_impure_bulkdock_input.csv $BULK/INPUTS/<target_name>_impure_knitwork.csv
```


## Run Placement Jobs 

Can be done here or directly from the terminal by removing the !

In [ ]:
!python -m bulkdock place <TARGET_NAME> <target_name>_fragmenstein.csv --split 2000

In [ ]:
!python -m bulkdock place <TARGET_NAME> <target_name>_pure_knitwork.csv --split 2000

In [ ]:
!python -m bulkdock place <TARGET_NAME> <target_name>_impure_knitwork.csv --split 2000

## Export Placed Poses for GNINA Minimisation

In [ ]:
%load_ext autoreload
%autoreload 2
import hippo
import hippo.pset as pset
pset.mp = mp
import mrich
from mrich import print
from pathlib import Path
from os import environ
import shutil
import molparse as mp
import plotly.express as px



In [ ]:
animal = hippo.HIPPO(target_name, bulk_target_dir / f"{target_name}.sqlite")

In [ ]:
animal.tags #Display available tags

In [ ]:
fragmenstein_poses = animal.poses.get_by_tag(<target_name>_fragmenstein) #replace this with the relevant tag associated with your fragmenstein poses.

fragmenstein_poses.to_fragalysis(
    str(gnina_inputs_fragmenstein_dir / f"{target_name}_bulkdock_poses_fragmenstein.sdf"),
    method = "fragmenstein",
    submitter_name = "YOUR NAME HERE",
    submitter_email = "YOUR EMAIL HERE",
    submitter_institution = "YOUR INSTITUTION HERE",
    copy_reference_pdbs=True
)

In [ ]:
knitwork_pure_poses = animal.poses.get_by_tag(<target_name>_pure_knitwork) #replace this with the relevant tag associated with your pure knitwork poses.

knitwork_pure_poses.to_fragalysis(
    str(gnina_inputs_knitwork_pure_dir / f"{target_name}_bulkdock_poses_knitwork_pure.sdf"),
    method = "knitwork",
    submitter_name = "YOUR NAME HERE",
    submitter_email = "YOUR EMAIL HERE",
    submitter_institution = "YOUR INSTITUTION HERE",
    copy_reference_pdbs=True
)

In [ ]:
knitwork_impure_poses = animal.poses.get_by_tag(<target_name>_impure_knitwork) #replace this with the relevant tag associated with your impure knitwork poses.

knitwork_impure_poses.to_fragalysis(
    str(gnina_inputs_knitwork_impure_dir / f"{target_name}_bulkdock_poses_knitwork_impure.sdf"),
    method = "knitwork",
    submitter_name = "YOUR NAME HERE",
    submitter_email = "YOUR EMAIL HERE",
    submitter_institution = "YOUR INSTITUTION HERE",
    copy_reference_pdbs=True
)

## Run GNINA Minimisation and Scoring

In [ ]:
wrap_command = f"""singularity exec --nv \
--bind /opt/xchem-fragalysis-2:/opt/xchem-fragalysis-2 \
/opt/xchem-fragalysis-2/XChem-FFF/openbind-rescore/gnina/gnina_singularity.sif \
python -u /opt/xchem-fragalysis-2/XChem-FFF/openbind-rescore/gnina_rescore.py \
--gnina_path /opt/xchem-fragalysis-2/XChem-FFF/openbind-rescore/gnina/gnina \
--input_ligands {str(gnina_inputs_fragmenstein_dir / f"{target_name}_bulkdock_poses_fragmenstein.sdf")} \
--ref_pdbs {str(gnina_inputs_fragmenstein_dir / f"{target_name}_bulkdock_poses_fragmenstein_refs.zip")} \
--output_path {gnina_outputs_fragmenstein_dir} \
--cpu_cores 8"""

!sbatch -p main -c 8 --wrap="{wrap_command}"

In [ ]:
wrap_command = f"""singularity exec --nv \
--bind /opt/xchem-fragalysis-2:/opt/xchem-fragalysis-2 \
/opt/xchem-fragalysis-2/XChem-FFF/openbind-rescore/gnina/gnina_singularity.sif \
python -u /opt/xchem-fragalysis-2/XChem-FFF/openbind-rescore/gnina_rescore.py \
--gnina_path /opt/xchem-fragalysis-2/XChem-FFF/openbind-rescore/gnina/gnina \
--input_ligands {str(gnina_inputs_knitwork_pure_dir / f"{target_name}_bulkdock_poses_knitwork_pure.sdf")} \
--ref_pdbs {str(gnina_inputs_knitwork_pure_dir / f"{target_name}_bulkdock_poses_knitwork_pure.zip")} \
--output_path {gnina_outputs_knitwork_pure_dir} \
--cpu_cores 8"""

!sbatch -p main -c 8 --wrap="{wrap_command}"

In [ ]:
wrap_command = f"""singularity exec --nv \
--bind /opt/xchem-fragalysis-2:/opt/xchem-fragalysis-2 \
/opt/xchem-fragalysis-2/XChem-FFF/openbind-rescore/gnina/gnina_singularity.sif \
python -u /opt/xchem-fragalysis-2/XChem-FFF/openbind-rescore/gnina_rescore.py \
--gnina_path /opt/xchem-fragalysis-2/XChem-FFF/openbind-rescore/gnina/gnina \
--input_ligands {str(gnina_inputs_knitwork_impure_dir / f"{target_name}_bulkdock_poses_knitwork_impure.sdf")} \
--ref_pdbs {str(gnina_inputs_knitwork_impure_dir / f"{target_name}_bulkdock_poses_knitwork_impure.zip")} \
--output_path {gnina_outputs_knitwork_impure_dir} \
--cpu_cores 8"""

!sbatch -p main -c 8 --wrap="{wrap_command}"

## Import GNINA minimised poses to HIPPO for visualisation.

In [ ]:
from pathlib import Path

sdf_dir = Path(gnina_outputs_fragmenstein_dir / "output_sdfs")

for sdf_file in sorted(sdf_dir.glob("*.sdf")):
    animal.load_sdf(
        target=target_name,
        path=sdf_file,
        compound_tags=["scaffold"],
        pose_tags=["gnina_repose", "fragmenstein"],
        name_col=None,
    )

In [ ]:
from pathlib import Path

sdf_dir = Path(gnina_outputs_knitwork_pure_dir / "output_sdfs")

for sdf_file in sorted(sdf_dir.glob("*.sdf")):
    animal.load_sdf(
        target=target_name,
        path=sdf_file,
        compound_tags=["scaffold"],
        pose_tags=["gnina_repose", "pure_knitwork"],
        name_col=None,
    )

In [ ]:
from pathlib import Path

sdf_dir = Path(gnina_outputs_knitwork_impure_dir / "output_sdfs")

for sdf_file in sorted(sdf_dir.glob("*.sdf")):
    animal.load_sdf(
        target=target_name,
        path=sdf_file,
        compound_tags=["scaffold"],
        pose_tags=["gnina_repose", "impure_knitwork"],
        name_col=None,
    )